# Model Inference - Crack Detection Prediction

This notebook loads the trained model and performs predictions on all samples in the `samples` directory.

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import torchvision.transforms as transforms

# Device configuration
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## Configuration

Set the paths for model, input samples, and output predictions

In [ ]:
# Paths configuration
MODEL_PATH = "Generalized_dataset.pt"  # Path to saved model
SAMPLES_DIR = "samples/"  # Directory containing input images
OUTPUT_DIR = "predictions/"  # Directory to save predictions
IMG_SIZE = (800, 800)  # Model input size

# Create output directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/masks", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/overlays", exist_ok=True)

print(f"Model path: {MODEL_PATH}")
print(f"Input samples: {SAMPLES_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

## Load the Trained Model

In [ ]:
# Load the model
try:
    model = torch.load(MODEL_PATH, map_location=DEVICE)
    model.eval()
    print("✅ Model loaded successfully!")
    
    # Count parameters
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Total Parameters: {num_params:,}")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    raise

## Define Preprocessing and Postprocessing Functions

In [ ]:
def preprocess_image(image_path, img_size=(800, 800)):
    """
    Load and preprocess an image for model input
    
    Args:
        image_path: Path to the image file
        img_size: Target size for the image
    
    Returns:
        - Original image (for visualization)
        - Preprocessed tensor ready for model
        - Original dimensions
    """
    # Load image
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Could not load image: {image_path}")
    
    # Store original dimensions
    original_size = img.shape
    
    # Resize to model input size
    img_resized = cv2.resize(img, img_size)
    
    # Convert to tensor and normalize
    img_tensor = torch.from_numpy(img_resized).float()
    img_tensor = img_tensor / 255.0  # Normalize to [0, 1]
    img_tensor = img_tensor.unsqueeze(0).unsqueeze(0)  # Add batch and channel dimensions
    
    return img, img_tensor, original_size


def postprocess_prediction(pred_tensor, original_size=None, threshold=0.5):
    """
    Convert model prediction to binary mask
    
    Args:
        pred_tensor: Model output tensor
        original_size: Original image size to resize back to
        threshold: Threshold for binary classification
    
    Returns:
        Binary mask as numpy array
    """
    # Remove batch and channel dimensions
    pred = pred_tensor.squeeze().cpu().detach().numpy()
    
    # Apply threshold
    pred_binary = (pred > threshold).astype(np.uint8) * 255
    
    # Resize back to original size if needed
    if original_size is not None:
        pred_binary = cv2.resize(pred_binary, (original_size[1], original_size[0]), 
                                interpolation=cv2.INTER_NEAREST)
    
    return pred_binary


def create_overlay(original_img, mask, alpha=0.5):
    """
    Create an overlay visualization of the prediction on the original image
    
    Args:
        original_img: Original grayscale image
        mask: Predicted binary mask
        alpha: Transparency factor for overlay
    
    Returns:
        Overlay image (BGR format for cv2.imwrite)
    """
    # Convert grayscale to BGR for visualization
    if len(original_img.shape) == 2:
        img_bgr = cv2.cvtColor(original_img, cv2.COLOR_GRAY2BGR)
    else:
        img_bgr = original_img.copy()
    
    # Resize mask to match original image if needed
    if mask.shape != original_img.shape:
        mask = cv2.resize(mask, (original_img.shape[1], original_img.shape[0]))
    
    # Create red overlay for detected cracks
    overlay = img_bgr.copy()
    overlay[mask > 127] = [0, 0, 255]  # Red color for cracks (BGR format)
    
    # Blend with original image
    result = cv2.addWeighted(img_bgr, 1-alpha, overlay, alpha, 0)
    
    return result

## Process All Samples

Run inference on all images in the samples directory

In [ ]:
# Get all image files from samples directory
image_extensions = ['.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp']
image_files = [f for f in os.listdir(SAMPLES_DIR) 
               if os.path.splitext(f)[1].lower() in image_extensions]

print(f"Found {len(image_files)} images to process")

if len(image_files) == 0:
    print("⚠️  No images found in samples directory!")
else:
    print("\nProcessing images...")
    
    # Process each image
    results = []
    
    with torch.no_grad():  # Disable gradient computation for inference
        for img_file in tqdm(image_files, desc="Processing"):
            try:
                # Full path to image
                img_path = os.path.join(SAMPLES_DIR, img_file)
                
                # Preprocess
                original_img, img_tensor, original_size = preprocess_image(img_path, IMG_SIZE)
                
                # Move to device and predict
                img_tensor = img_tensor.to(DEVICE)
                prediction = model(img_tensor)
                
                # Postprocess
                pred_mask = postprocess_prediction(prediction, original_size, threshold=0.5)
                
                # Create overlay
                overlay = create_overlay(original_img, pred_mask, alpha=0.4)
                
                # Save results
                base_name = os.path.splitext(img_file)[0]
                
                # Save mask
                mask_path = os.path.join(OUTPUT_DIR, "masks", f"{base_name}_mask.png")
                cv2.imwrite(mask_path, pred_mask)
                
                # Save overlay
                overlay_path = os.path.join(OUTPUT_DIR, "overlays", f"{base_name}_overlay.png")
                cv2.imwrite(overlay_path, overlay)
                
                # Store results for summary
                crack_percentage = (pred_mask > 127).sum() / pred_mask.size * 100
                results.append({
                    'filename': img_file,
                    'crack_percentage': crack_percentage,
                    'has_crack': crack_percentage > 0.1  # Consider >0.1% as having cracks
                })
                
            except Exception as e:
                print(f"\n❌ Error processing {img_file}: {e}")
                continue
    
    print(f"\n✅ Processing complete! Results saved to {OUTPUT_DIR}")

## Results Summary

In [ ]:
# Print summary statistics
if results:
    print("\n" + "="*60)
    print("PREDICTION SUMMARY")
    print("="*60)
    
    total_images = len(results)
    images_with_cracks = sum(1 for r in results if r['has_crack'])
    images_without_cracks = total_images - images_with_cracks
    
    print(f"\nTotal images processed: {total_images}")
    print(f"Images with detected cracks: {images_with_cracks} ({images_with_cracks/total_images*100:.1f}%)")
    print(f"Images without cracks: {images_without_cracks} ({images_without_cracks/total_images*100:.1f}%)")
    
    # Show top 5 images with most cracks
    print("\n" + "-"*60)
    print("Top 5 images with most detected cracks:")
    print("-"*60)
    sorted_results = sorted(results, key=lambda x: x['crack_percentage'], reverse=True)
    for i, result in enumerate(sorted_results[:5], 1):
        print(f"{i}. {result['filename']}: {result['crack_percentage']:.2f}% crack coverage")
    
    print("\n" + "="*60)

## Visualize Sample Predictions

Display a few sample predictions

In [ ]:
# Visualize some sample predictions
num_samples = min(6, len(image_files))

if num_samples > 0:
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for idx in range(num_samples):
        img_file = image_files[idx]
        base_name = os.path.splitext(img_file)[0]
        
        # Load original image
        original_path = os.path.join(SAMPLES_DIR, img_file)
        original = cv2.imread(original_path, cv2.IMREAD_GRAYSCALE)
        
        # Load predicted mask
        mask_path = os.path.join(OUTPUT_DIR, "masks", f"{base_name}_mask.png")
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
        # Load overlay
        overlay_path = os.path.join(OUTPUT_DIR, "overlays", f"{base_name}_overlay.png")
        overlay = cv2.imread(overlay_path)
        overlay_rgb = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)
        
        # Display
        axes[idx, 0].imshow(original, cmap='gray')
        axes[idx, 0].set_title(f"Original: {img_file}")
        axes[idx, 0].axis('off')
        
        axes[idx, 1].imshow(mask, cmap='gray')
        axes[idx, 1].set_title("Predicted Mask")
        axes[idx, 1].axis('off')
        
        axes[idx, 2].imshow(overlay_rgb)
        axes[idx, 2].set_title("Overlay")
        axes[idx, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/visualization_samples.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✅ Visualization saved to {OUTPUT_DIR}/visualization_samples.png")

## Export Results to CSV (Optional)

In [ ]:
# Export results to CSV file
import csv

if results:
    csv_path = os.path.join(OUTPUT_DIR, "prediction_results.csv")
    
    with open(csv_path, 'w', newline='') as csvfile:
        fieldnames = ['filename', 'crack_percentage', 'has_crack']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        
        writer.writeheader()
        for result in results:
            writer.writerow(result)
    
    print(f"✅ Results exported to {csv_path}")